In [ ]:
# Weight Optimization

# NOTE:best_tickers -> selected
# cad_prices -> aligned_prices

# for subsequent historical calculations.
historical_prices = cad_prices[best_tickers]

# For each stock try transferring 1% point to every other stock and keep the lowest std


def weighted_std(weights): 
    shares = weights / historical_prices.iloc[0] 
    values = (historical_prices * shares).sum(axis=1)
    returns = values.pct_change().iloc[1:]
    return returns.std(ddof=1)

target_weights = initial_weights.copy() 
step = 0.01  
current_std = weighted_std(target_weights)  
move_history = [] 

improved = True  
while improved: 
    improved = False  # Assume no improvement at the beginning of each iteration

    for a in best_tickers:  # Try each selected stock as the donor
        if target_weights[a] <= step:  
            continue

        best_b = None  
        lowest_std = current_std  

        for b in best_tickers:  
            if a == b:  
                continue

            trial = target_weights.copy()  
            trial[a] -= step  
            trial[b] += step 

            if trial.max() > 0.15: 
                continue
            if trial.groupby(sectors).sum().max() > 0.40:  
                continue
                # check restrictions

            trial_std = weighted_std(trial)  # calculate portfolio risk after transfer
            if trial_std < lowest_std:
                best_b = b
                lowest_std = trial_std

        if best_b is not None:  
            target_weights[a] -= step
            target_weights[best_b] += step
            current_std = lowest_std
            move_history.append([a, best_b, current_std])  
            improved = True  

print(f"Optimized Std: {current_std:.4%}")  # Standard deviation after weight optimization

In [ ]:
#buy_and_hold
def buy_hold_std(names): 
    prices = cad_prices[names]
    shares = (1 / len(names)) / prices.iloc[0] 
    portfolio_value = (prices * shares).sum(axis=1)
    returns = portfolio_value.pct_change().iloc[1:]
    return returns.std(ddof=1) 